# 📊 DIAM-IA — Agriculture & Déforestation au Gabon
### Notebook Google Colab — rapport complet (FAOSTAT + Earth Engine)

**Objectif** : préparer l'ensemble des données et visualisations *interactives* (fichiers `.html`) de l'article DIAM-IA « Agriculture & Déforestation au Gabon ».

**Points clés de ce notebook :**
- Toutes les figures sont générées avec **Plotly** ou **geemap** et exportées en **HTML interactif** — aucune image statique (`.png`).
- Aucun chiffre n'est jamais inventé : les seuils de classification (Module G) sont calculés à partir des données elles-mêmes (médiane), pas fixés arbitrairement.
- Le projet Earth Engine utilisé est **`gee-gabon-project`**.
- Bugs corrigés par rapport à la version précédente : groupement des classes WorldCover (ordre des bandes), valeurs `None` non protégées (`.get('area', 0)`), classification de quadrant jamais réellement appliquée, module Hansen manquant, grille jamais rapatriée en DataFrame.

**Avant de lancer ce notebook dans Colab :**
1. Téléversez le fichier FAOSTAT (`Production_Crops_Livestock_E_All_Data_Normalized.csv`, téléchargeable sur [fao.org/faostat](https://www.fao.org/faostat/en/#data/QCL)) dans l'environnement Colab — voir Cellule 2.
2. Assurez-vous que le projet `gee-gabon-project` existe bien dans votre compte Google Cloud et qu'Earth Engine y est activé.
3. Le Module F dépend d'un asset Earth Engine tiers (WRI) dont la disponibilité peut varier — voir la note dans ce module si vous obtenez une erreur d'accès.


## 🔹 Cellule 1 — Installation & configuration

In [ ]:
# Earth Engine + geemap pour la carte interactive
# Plotly pour les graphiques interactifs (déjà préinstallé sur Colab, on force la version récente par sécurité)
!pip install -q geemap earthengine-api plotly --upgrade

import ee
import geemap
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)


In [ ]:
# Authentification + initialisation Earth Engine
# ⚠️ Remplacez 'gee-gabon-project' si vous utilisez un autre projet Google Cloud.
ee.Authenticate()
ee.Initialize(project='gee-gabon-project')

gabon = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq('ADM0_NAME', 'Gabon'))
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Gabon'))
geometry = gabon.geometry()
noms_provinces = provinces.aggregate_array('ADM1_NAME').getInfo()

print(f"✅ Earth Engine initialisé sur le projet 'gee-gabon-project'")
print(f"✅ {len(noms_provinces)} provinces chargées : {noms_provinces}")


## 🔹 Cellule 2 — Import du fichier FAOSTAT

Téléchargez au préalable le fichier **"Crops and livestock products"** (format *bulk download*, normalisé) depuis
[fao.org/faostat/en/#data/QCL](https://www.fao.org/faostat/en/#data/QCL), puis téléversez-le ci-dessous.

In [ ]:
from google.colab import files

print("Sélectionnez le fichier FAOSTAT (.csv) à téléverser :")
uploaded = files.upload()
faostat_filename = list(uploaded.keys())[0]
print(f"✅ Fichier reçu : {faostat_filename}")


## 🔹 Module A — Préparation des données FAOSTAT

In [ ]:
def prepare_faostat(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(" ", "_").str.replace("(", "").str.replace(")", "")
    )
    return df

df_raw = pd.read_csv(faostat_filename, encoding='latin1', low_memory=False)
df = prepare_faostat(df_raw)

gabon_df = df[df['area'].eq('Gabon')].copy()

# Principales cultures gabonaises — à ajuster si besoin selon les cultures disponibles dans le fichier
cultures = [
    'Cassava, fresh', 'Plantains and cooking bananas', 'Bananas',
    'Maize (corn)', 'Cocoa beans', 'Oil palm fruit', 'Natural rubber in primary forms',
]

agri = gabon_df[gabon_df['item'].isin(cultures)].copy()
print(f"✅ {len(agri)} lignes sélectionnées pour {len(cultures)} cultures")

cultures_absentes = set(cultures) - set(agri['item'].unique())
if cultures_absentes:
    print(f"⚠️ Cultures absentes du fichier FAOSTAT : {cultures_absentes}")


## 🔹 Module B — Production = Surface × Rendement

On extrait séparément les trois séries (`Production`, `Area harvested`, `Yield`) afin de pouvoir, plus loin,
déterminer si la croissance de la production vient d'une extension des surfaces ou d'une amélioration du rendement.

In [ ]:
def extraire_element(df, element_nom, agg='sum'):
    sub = df[df['element'].eq(element_nom)].copy()
    sub['value'] = pd.to_numeric(sub['value'], errors='coerce')
    return sub.pivot_table(index='year', columns='item', values='value', aggfunc=agg)

production_pivot = extraire_element(agri, 'Production', agg='sum')
surface_pivot    = extraire_element(agri, 'Area harvested', agg='sum')
rendement_pivot  = extraire_element(agri, 'Yield', agg='mean')

print("✅ Production, Surface et Rendement extraits séparément")
production_pivot.tail()


## 🔹 Figures interactives (HTML)

Chaque figure est exportée en fichier `.html` autonome, prêt à être intégré dans l'article via une balise `<iframe>`
(voir le template d'article fourni séparément).

In [ ]:
# Figure 1 — Production (interactive)
fig1 = px.line(
    production_pivot.reset_index().melt(id_vars='year', var_name='Culture', value_name='Production (t)'),
    x='year', y='Production (t)', color='Culture', markers=True,
    title="Figure 1 — Production des principales cultures, Gabon (FAOSTAT)",
)
fig1.update_layout(
    template='plotly_white', font=dict(family='Arial', size=13),
    legend=dict(orientation='v'), hovermode='x unified',
    xaxis_title="Année", yaxis_title="Production (tonnes)",
)
fig1.write_html('fig1_production.html', include_plotlyjs='cdn')
fig1.show()


In [ ]:
# Figure 2 — Rendement (interactive)
fig2 = px.line(
    rendement_pivot.reset_index().melt(id_vars='year', var_name='Culture', value_name='Rendement (kg/ha)'),
    x='year', y='Rendement (kg/ha)', color='Culture', markers=True,
    title="Figure 2 — Rendement des principales cultures, Gabon (FAOSTAT)",
)
fig2.update_layout(
    template='plotly_white', font=dict(family='Arial', size=13),
    hovermode='x unified', xaxis_title="Année", yaxis_title="Rendement (kg/ha)",
)
fig2.write_html('fig2_rendement.html', include_plotlyjs='cdn')
fig2.show()


In [ ]:
# Figure 3 — Surface récoltée (interactive)
fig3 = px.line(
    surface_pivot.reset_index().melt(id_vars='year', var_name='Culture', value_name='Surface (ha)'),
    x='year', y='Surface (ha)', color='Culture', markers=True,
    title="Figure 3 — Surface récoltée par culture, Gabon (FAOSTAT)",
)
fig3.update_layout(
    template='plotly_white', font=dict(family='Arial', size=13),
    hovermode='x unified', xaxis_title="Année", yaxis_title="Surface (ha)",
)
fig3.write_html('fig3_surface.html', include_plotlyjs='cdn')
fig3.show()


## 🔹 Analyse de décomposition — surface ou rendement ?

Pour chaque culture, la production peut croître de deux façons : l'extension des surfaces cultivées, ou
l'amélioration du rendement à surface constante (intensification). Cette table répond directement à la
question : *la croissance vient-elle de la surface (pression potentielle sur la forêt) ou du rendement
(intensification) ?*

In [ ]:
resultats_decomposition = []

for culture in production_pivot.columns:
    prod = production_pivot[culture].dropna()
    surf = surface_pivot[culture].dropna() if culture in surface_pivot else pd.Series(dtype=float)
    rdt  = rendement_pivot[culture].dropna() if culture in rendement_pivot else pd.Series(dtype=float)

    if len(prod) < 2:
        continue  # pas assez de points pour calculer une croissance

    prod_debut, prod_fin = prod.iloc[0], prod.iloc[-1]
    surf_debut, surf_fin = (surf.iloc[0], surf.iloc[-1]) if len(surf) >= 2 else (np.nan, np.nan)
    rdt_debut, rdt_fin   = (rdt.iloc[0], rdt.iloc[-1]) if len(rdt) >= 2 else (np.nan, np.nan)

    croissance_prod = ((prod_fin / prod_debut) - 1) * 100 if prod_debut else np.nan
    croissance_surf = ((surf_fin / surf_debut) - 1) * 100 if surf_debut else np.nan
    croissance_rdt  = ((rdt_fin / rdt_debut) - 1) * 100 if rdt_debut else np.nan

    if pd.notna(croissance_surf) and pd.notna(croissance_rdt):
        moteur = 'Surface' if abs(croissance_surf) > abs(croissance_rdt) else 'Rendement'
    else:
        moteur = 'Non déterminable'

    resultats_decomposition.append({
        'Culture': culture,
        'Croissance production (%)': round(croissance_prod, 1) if pd.notna(croissance_prod) else None,
        'Croissance surface (%)': round(croissance_surf, 1) if pd.notna(croissance_surf) else None,
        'Croissance rendement (%)': round(croissance_rdt, 1) if pd.notna(croissance_rdt) else None,
        'Moteur principal': moteur,
    })

df_decomposition = pd.DataFrame(resultats_decomposition)
df_decomposition.to_csv('decomposition_production.csv', index=False)
df_decomposition


In [ ]:
# Figure 4 — Décomposition (graphique en barres groupées, interactif)
fig4 = go.Figure()
fig4.add_bar(name='Surface (%)', x=df_decomposition['Culture'], y=df_decomposition['Croissance surface (%)'])
fig4.add_bar(name='Rendement (%)', x=df_decomposition['Culture'], y=df_decomposition['Croissance rendement (%)'])
fig4.update_layout(
    barmode='group', template='plotly_white', font=dict(family='Arial', size=13),
    title="Figure 4 — Décomposition de la croissance de production : surface vs rendement",
    xaxis_title="Culture", yaxis_title="Croissance (%)", xaxis_tickangle=-30,
)
fig4.write_html('fig4_decomposition.html', include_plotlyjs='cdn')
fig4.show()


## 🔹 Module C — Occupation des sols (ESA WorldCover)

Avant de parler de déforestation, il faut d'abord savoir ce que le satellite voit *aujourd'hui* : quelle part
du territoire est en forêt, en terres cultivées, en zones humides, etc. ESA WorldCover fournit cette
cartographie à 10m de résolution (précision globale ≈ 76,7%, version 2021).

In [ ]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

classes_noms = {
    10: 'Forêt', 20: 'Arbustes', 30: 'Prairies', 40: 'Terres cultivées',
    50: 'Zones bâties', 60: 'Sol nu', 70: 'Neige/Glace', 80: 'Eau',
    90: 'Zones humides', 95: 'Mangrove', 100: 'Mousses/Lichens',
}

def calculer_surfaces(image_bande, region, scale=100):
    """
    Calcule la surface (km²) par classe d'occupation des sols.

    ⚠️ Bug corrigé : ee.Reducer.sum().group(groupField=N) attend que la bande
    de GROUPEMENT (la classe) soit placée en DERNIER, et groupField=N pointe
    l'index de cette bande. L'ordre correct est donc [valeur, classe], pas
    [classe, valeur] — sinon Earth Engine sommera la mauvaise bande.
    """
    pixel_area = ee.Image.pixelArea().divide(1e6)  # m² -> km²
    image_avec_surface = pixel_area.addBands(image_bande)  # bandes : [area(0), Map(1)]

    stats = image_avec_surface.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='classe'),  # groupField=1 -> bande 'Map'
        geometry=region.geometry(), scale=scale, maxPixels=1e10,
    )
    return stats

resultats = calculer_surfaces(worldcover.select('Map'), gabon).get('groups').getInfo()
total = sum(g['sum'] for g in resultats)

df_occupation = pd.DataFrame([
    {
        'Classe': classes_noms.get(g['classe'], f"Classe {g['classe']}"),
        'Surface_km2': round(g['sum'], 0),
        'Pourcentage': round((g['sum'] / total) * 100, 2),
    }
    for g in resultats
]).sort_values('Surface_km2', ascending=False)

print(df_occupation.to_string(index=False))
df_occupation.to_csv('occupation_sols_gabon.csv', index=False)


In [ ]:
# Figure 5 — Carte interactive de l'occupation des sols
wc_vis = {
    'bands': ['Map'], 'min': 10, 'max': 100,
    'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4',
                'f0f0f0', '0064c8', '0096a0', '00cf75', 'fae6a0'],
}

Map = geemap.Map(center=[-0.8, 11.6], zoom=6)
Map.addLayer(worldcover.clip(gabon), wc_vis, 'Occupation des sols 2021')
Map.addLayer(provinces.style(color='white', fillColor='00000000', width=1), {}, 'Provinces')
Map.add_legend(title="Occupation des sols", builtin_legend='ESA_WorldCover')
Map.to_html('fig5_occupation_sols.html')

print("✅ Carte interactive exportée : fig5_occupation_sols.html")
Map


In [ ]:
# Figure 6 — Répartition de l'occupation des sols (interactive)
fig6 = px.bar(
    df_occupation, x='Surface_km2', y='Classe', orientation='h',
    text='Pourcentage', title="Figure 6 — Occupation des sols au Gabon (ESA WorldCover 2021)",
)
fig6.update_traces(texttemplate='%{text}%', textposition='outside')
fig6.update_layout(
    template='plotly_white', xaxis_title="Surface (km²)", yaxis_title="",
    yaxis=dict(autorange='reversed'),
)
fig6.write_html('fig6_occupation_repartition.html', include_plotlyjs='cdn')
fig6.show()


## 🔹 Module D — Perte de couverture arborée (Hansen Global Forest Change)

⚠️ **Précaution méthodologique à conserver dans l'article** : des changements dans la méthode de détection
Hansen affectent la comparabilité de la série avant/après 2015. Les deux périodes ne doivent pas être
comparées directement en valeur absolue — c'est pourquoi la Figure 7 les distingue visuellement.

In [ ]:
hansen = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")

def calculate_loss(year):
    loss_year = hansen.select('lossyear').eq(year - 2000)
    area = ee.Image.pixelArea().updateMask(loss_year).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geometry, scale=30, maxPixels=1e13,
    )
    return ee.Feature(None, {'year': year, 'loss_m2': area.get('area')})

years = list(range(2001, 2024))
features = ee.FeatureCollection([calculate_loss(y) for y in years])
results = features.getInfo()

loss_df = pd.DataFrame([f['properties'] for f in results['features']])
# Bug corrigé : une année sans perte détectée renvoie loss_m2 = None (pas 0) ->
# sans ce fillna, la conversion km2 et les graphiques afficheraient des trous (NaN).
loss_df['loss_m2'] = loss_df['loss_m2'].fillna(0)
loss_df['loss_km2'] = loss_df['loss_m2'] / 1e6
loss_df.to_csv('perte_annuelle_foret.csv', index=False)

print(loss_df.to_string(index=False))


In [ ]:
# Figure 7 — Perte annuelle (interactive), avec repère de la rupture méthodologique 2015
fig7 = go.Figure()
fig7.add_bar(
    x=loss_df['year'], y=loss_df['loss_km2'],
    marker_color=['#94A3B8' if y < 2015 else '#DC2626' for y in loss_df['year']],
    name='Perte annuelle',
)
fig7.add_vline(x=2014.5, line_dash='dash', line_color='black')
fig7.add_annotation(
    x=2015, y=loss_df['loss_km2'].max() * 0.95,
    text="Rupture méthodologique<br>Hansen (post-2015)",
    showarrow=False, font=dict(size=11), align='left', xanchor='left',
)
fig7.update_layout(
    template='plotly_white',
    title="Figure 7 — Perte annuelle de couverture arborée, Gabon (2001-2023)",
    xaxis_title="Année", yaxis_title="km²",
)
fig7.write_html('fig7_perte_annuelle.html', include_plotlyjs='cdn')
fig7.show()


## 🔹 Module E — Perte forestière par province

Calcul indépendant pour chaque province — **l'interprétation ne sera rédigée qu'après lecture du tableau**,
jamais avant.

In [ ]:
resultats_provinces = []

for nom in noms_provinces:
    province = provinces.filter(ee.Filter.eq('ADM1_NAME', nom))
    perte_area = ee.Image.pixelArea().updateMask(hansen.select('loss')).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=province.geometry(), scale=30, maxPixels=1e13,
    )
    # Bug corrigé : .get('area', 0) ne protège PAS contre une valeur None (la clé existe
    # quand même). Sans le "or 0", une province sans perte détectée aurait affiché "None".
    surface = (perte_area.getInfo().get('area') or 0) / 1e6
    resultats_provinces.append({'Province': nom, 'Perte_totale_km2': round(surface, 1)})
    print(f"  {nom} : {surface:.1f} km²")

df_provinces = pd.DataFrame(resultats_provinces).sort_values('Perte_totale_km2', ascending=False)
df_provinces.to_csv('perte_par_province.csv', index=False)
df_provinces


In [ ]:
# Figure 9 — Perte forestière cumulée par province (interactive)
fig9 = px.bar(
    df_provinces, x='Perte_totale_km2', y='Province', orientation='h',
    title="Figure 9 — Perte forestière cumulée par province (2001-2023)",
)
fig9.update_traces(marker_color='#3B82F6')
fig9.update_layout(
    template='plotly_white', xaxis_title="km²", yaxis_title="",
    yaxis=dict(autorange='reversed'),
)
fig9.write_html('fig9_perte_provinces.html', include_plotlyjs='cdn')
fig9.show()


## 🔹 Module F — Facteurs de perte forestière (WRI / Google DeepMind)

⚠️ Si l'asset `projects/wri-datalab/TCL_drivers/tcl_drivers_2023` n'est pas accessible publiquement au moment
de l'exécution, cherchez la version publiée sur **Global Forest Watch / World Resources Institute — "Tree
cover loss by dominant driver"**, téléchargeable en GeoTIFF et importable comme asset Earth Engine
personnel.

In [ ]:
drivers = ee.Image("projects/wri-datalab/TCL_drivers/tcl_drivers_2023")

driver_classes = {
    1: 'Agriculture permanente', 2: 'Agriculture itinérante', 3: 'Exploitation forestière (logging)',
    4: 'Exploitation minière', 5: 'Infrastructures', 6: 'Incendies', 7: 'Perturbation naturelle',
}

def surface_par_driver(region):
    resultats = []
    for code_, nom in driver_classes.items():
        masque = drivers.eq(code_)
        area = ee.Image.pixelArea().updateMask(masque).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=region, scale=1000, maxPixels=1e13,
        )
        # Même correction que Module E : "or 0" contre les None
        surface = (area.getInfo().get('area') or 0) / 1e6
        resultats.append({'Facteur': nom, 'Surface_km2': round(surface, 1)})
    return pd.DataFrame(resultats).sort_values('Surface_km2', ascending=False)

df_drivers = surface_par_driver(geometry)
df_drivers.to_csv('facteurs_perte_foret.csv', index=False)
df_drivers


In [ ]:
# Figure 8 — Facteurs de perte de couverture arborée (interactive)
fig8 = px.bar(
    df_drivers, x='Surface_km2', y='Facteur', orientation='h',
    title="Figure 8 — Facteurs de perte de couverture arborée, Gabon (2001-2023)",
)
fig8.update_traces(marker_color='#7C3AED')
fig8.update_layout(
    template='plotly_white', xaxis_title="km²", yaxis_title="",
    yaxis=dict(autorange='reversed'),
)
fig8.write_html('fig8_facteurs_perte.html', include_plotlyjs='cdn')
fig8.show()


## 🔹 Module G — Matrice Agriculture × Forêt (la figure signature)

On croise, cellule par cellule (grille ~10km), l'intensité de l'agriculture (ESA WorldCover, classe
"Terres cultivées") et la perte forestière (Hansen), pour distinguer 4 profils de territoire.

⚠️ **Calcul potentiellement long** selon la taille de la grille. Pour un premier test rapide, passez la
grille en 25km (`geometry.coveringGrid('EPSG:4326', 25000)`) avant de repasser en 10km pour la version
finale.

In [ ]:
cropland_mask = worldcover.select('Map').eq(40)   # Terres cultivées
loss_mask     = hansen.select('loss')

TAILLE_CELLULE_M = 10000  # 10km — réduire à 25000 pour un premier test plus rapide
grille = geometry.coveringGrid('EPSG:4326', TAILLE_CELLULE_M)

def analyser_cellule(feature):
    geom = feature.geometry()
    centroid = geom.centroid(1)

    crop_area = ee.Image.pixelArea().updateMask(cropland_mask).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geom, scale=100, maxPixels=1e9).get('area')
    loss_area = ee.Image.pixelArea().updateMask(loss_mask).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geom, scale=30, maxPixels=1e9).get('area')

    return feature.set({
        # ee.Number(None) lèverait une erreur silencieuse côté serveur : on protège avec unmask(0)
        'crop_km2': ee.Number(crop_area).unmask(0).divide(1e6),
        'loss_km2': ee.Number(loss_area).unmask(0).divide(1e6),
        'lon': centroid.coordinates().get(0),
        'lat': centroid.coordinates().get(1),
    })

grille_analysee = grille.map(analyser_cellule)

n_cellules = grille.size().getInfo()
print(f"Nombre de cellules dans la grille : {n_cellules}")


In [ ]:
# Extraction de la grille analysée en DataFrame local
# (⚠️ ce qui manquait dans la version précédente : grille_analysee restait
#  côté serveur Earth Engine et n'était jamais rapatriée en local)
df_grille = geemap.ee_to_df(grille_analysee)
df_grille = df_grille.dropna(subset=['crop_km2', 'loss_km2', 'lon', 'lat'])
print(f"✅ {len(df_grille)} cellules extraites avec succès")
df_grille.head()


In [ ]:
# Seuils de classification — définis à partir des données elles-mêmes
# (médiane des cellules concernées), jamais fixés arbitrairement.
seuil_agri  = df_grille.loc[df_grille['crop_km2']  > 0, 'crop_km2'].median()
seuil_perte = df_grille.loc[df_grille['loss_km2'] > 0, 'loss_km2'].median()

print(f"Seuil agriculture  (médiane des cellules avec agriculture) : {seuil_agri:.3f} km²")
print(f"Seuil perte forêt  (médiane des cellules avec perte)       : {seuil_perte:.3f} km²")

def classer_quadrant(row, seuil_agri, seuil_perte):
    if row['crop_km2'] < seuil_agri and row['loss_km2'] < seuil_perte:
        return "Zone stable (faible agriculture, faible perte)"
    elif row['crop_km2'] >= seuil_agri and row['loss_km2'] < seuil_perte:
        return "🟢 Potentiel agricole (agriculture sans perte forestière)"
    elif row['crop_km2'] < seuil_agri and row['loss_km2'] >= seuil_perte:
        return "🔴 Zone sensible (perte forestière sans agriculture visible)"
    else:
        return "🟠 Zone à surveiller (agriculture + perte forestière)"

# Bug corrigé : cette ligne était laissée en commentaire (donc jamais exécutée)
# dans la version précédente — la classification n'était en réalité jamais appliquée.
df_grille['quadrant'] = df_grille.apply(lambda r: classer_quadrant(r, seuil_agri, seuil_perte), axis=1)

df_grille.to_csv('grille_quadrants.csv', index=False)
print(df_grille['quadrant'].value_counts())


In [ ]:
# Figure 10 (signature DIAM-IA) — Carte interactive des 4 quadrants
color_map = {
    "Zone stable (faible agriculture, faible perte)": "#94A3B8",
    "🟢 Potentiel agricole (agriculture sans perte forestière)": "#10B981",
    "🔴 Zone sensible (perte forestière sans agriculture visible)": "#DC2626",
    "🟠 Zone à surveiller (agriculture + perte forestière)": "#F59E0B",
}

fig10 = px.scatter_mapbox(
    df_grille, lat='lat', lon='lon', color='quadrant',
    color_discrete_map=color_map,
    hover_data={'crop_km2': ':.2f', 'loss_km2': ':.2f', 'lat': False, 'lon': False},
    zoom=5.2, center={'lat': -0.8, 'lon': 11.6}, height=650,
    title="Figure 10 — Matrice Agriculture × Forêt, par cellule de 10km (Gabon)",
)
# style 'carto-positron' : fond de carte ouvert, sans clé Mapbox nécessaire
fig10.update_layout(mapbox_style='carto-positron', margin=dict(l=0, r=0, t=50, b=0))
fig10.write_html('fig10_matrice_agriculture_foret.html', include_plotlyjs='cdn')
fig10.show()


> **➡️ L'interprétation de cette carte se rédige uniquement après avoir vu les résultats** — jamais
> avant. C'est cette figure qui distingue une étude data d'un simple constat : elle identifie concrètement
> où l'agriculture peut se développer sans coût forestier (quadrant vert), et où une investigation
> complémentaire est nécessaire (quadrant rouge).

## 🔹 Récupérer les fichiers générés

Tous les fichiers HTML interactifs (et le CSV de décomposition) sont prêts dans l'environnement Colab.
Téléchargez-les pour les intégrer au template d'article `article_donnees_agricoles_gabon.html`
(chaque figure y est déjà référencée dans une balise `<iframe>`).

In [ ]:
from google.colab import files

fichiers_a_telecharger = [
    'fig1_production.html', 'fig2_rendement.html', 'fig3_surface.html',
    'fig4_decomposition.html', 'fig5_occupation_sols.html', 'fig6_occupation_repartition.html',
    'fig7_perte_annuelle.html', 'fig8_facteurs_perte.html', 'fig9_perte_provinces.html',
    'fig10_matrice_agriculture_foret.html',
    'decomposition_production.csv', 'occupation_sols_gabon.csv', 'perte_annuelle_foret.csv',
    'perte_par_province.csv', 'facteurs_perte_foret.csv', 'grille_quadrants.csv',
]

for f in fichiers_a_telecharger:
    try:
        files.download(f)
    except Exception as e:
        print(f"⚠️ Impossible de télécharger {f} : {e}")


---
### ✅ Prochaine étape

Ouvrez `article_donnees_agricoles_gabon.html`, placez les fichiers `fig1_production.html` à
`fig_carte_agricole.html` dans le même dossier que l'article, et remplacez les valeurs `[À insérer]`
des tableaux par les résultats réels une fois ce notebook exécuté avec vos données FAOSTAT.